### Qwen2.5-7B-Instruct

In [1]:
!pip install -q transformers accelerate h5py huggingface_hub scikit-learn

In [2]:
!nvidia-smi

Sat Apr 25 19:10:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   33C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os, sys

REPO_DIR = '/content/emotion-mechanisms-llm'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/daspushpita/emotion-mechanisms-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

for p in [f'{REPO_DIR}/src', f'{REPO_DIR}/scripts']:
    if p not in sys.path:
        sys.path.insert(0, p)

Cloning into '/content/emotion-mechanisms-llm'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 143 (delta 69), reused 114 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.12 MiB | 19.03 MiB/s, done.
Resolving deltas: 100% (69/69), done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Patch config BEFORE importing build_emotion_vectors so its module-level
# `from emotion_mechanisms.config import ...` picks up the Drive paths.
import emotion_mechanisms.config as cfg
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/emotion-mechanisms-llm')  # edit if needed

cfg.EMOTIONAL_STORIES_DATASET = DATA_ROOT / "datasets/processed/emotional_stories_qwen32B_v1.jsonl"
cfg.NEUTRAL_STORIES_DATASET   = DATA_ROOT / "datasets/processed/neutral_stories_qwen32B_v1.jsonl"
cfg.ACTIVATIONS_PATH          = DATA_ROOT / "results/activations/activations_7b.h5"
cfg.ANALYSIS_MODEL_7B         = "Qwen/Qwen2.5-7B-Instruct"
cfg.TOKEN_POSITION            = "mean"

assert cfg.EMOTIONAL_STORIES_DATASET.exists(), f'Missing: {cfg.EMOTIONAL_STORIES_DATASET}'
assert cfg.NEUTRAL_STORIES_DATASET.exists(),   f'Missing: {cfg.NEUTRAL_STORIES_DATASET}'
cfg.ACTIVATIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Datasets found. Output ->', cfg.ACTIVATIONS_PATH)

Datasets found. Output -> /content/drive/MyDrive/emotion-mechanisms-llm/results/activations/activations_7b.h5


In [6]:
from huggingface_hub import notebook_login
notebook_login()

In [6]:
import importlib
import build_emotion_vectors
importlib.reload(build_emotion_vectors)

build_emotion_vectors.main(max_stories=1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Neutral stories: 100%|██████████| 1/1 [00:00<00:00, 13.09it/s]


In [7]:
import h5py
with h5py.File(cfg.ACTIVATIONS_PATH, 'r') as f:
    print('Top-level keys:', list(f.keys()))
    print('Emotions stored:', list(f['emotional'].keys()))
    first_layer = list(f['emotional/happy'].keys())[0]
    print(f'emotional/happy/{first_layer} shape:', f[f'emotional/happy/{first_layer}'].shape)

Top-level keys: ['emotional', 'neutral']
Emotions stored: ['afraid', 'angry', 'calm', 'desperate', 'guilty', 'happy', 'inspired', 'loving', 'nervous', 'proud', 'sad', 'surprised']
emotional/happy/layer_0 shape: (1, 3584)


In [10]:
import numpy as np
import h5py

with h5py.File(cfg.ACTIVATIONS_PATH, 'r') as f:
    print(f"{'Emotion':<12} {'Layer':<10} {'Mean':>10} {'Std':>10} {'NaN':>6} {'Zero':>6}")
    print("-" * 56)
    for emotion in f['emotional'].keys():
        layers = list(f[f'emotional/{emotion}'].keys())
        # spot-check first, middle, last layer
        for layer in [layers[0], layers[len(layers)//2], layers[-1]]:
            arr = f[f'emotional/{emotion}/{layer}'][:]
            print(f"{emotion:<12} {layer:<10} {arr.mean():>10.3f} {arr.std():>10.3f} "
                f"{str(np.isnan(arr).any()):>6} {str((arr==0).all(-1).any()):>6}")
    
    # Also check neutral
    print("\nNeutral keys:", list(f['neutral'].keys()))
    n_layers = list(f['emotional/happy'].keys())
    print(f"Layers saved per emotion: {len(n_layers)} (e.g. {n_layers[0]} … {n_layers[-1]})")

Emotion      Layer            Mean        Std    NaN   Zero
--------------------------------------------------------
afraid       layer_0        -0.003      0.125  False  False
afraid       layer_20        0.000      2.277  False  False
afraid       layer_8        -0.023      1.590  False  False
angry        layer_0        -0.002      0.127  False  False
angry        layer_20       -0.013      2.519  False  False
angry        layer_8        -0.024      1.828  False  False
calm         layer_0        -0.003      0.128  False  False
calm         layer_20       -0.008      2.456  False  False
calm         layer_8        -0.021      1.635  False  False
desperate    layer_0        -0.003      0.126  False  False
desperate    layer_20       -0.008      2.325  False  False
desperate    layer_8        -0.024      1.562  False  False
guilty       layer_0        -0.002      0.129  False  False
guilty       layer_20       -0.007      2.714  False  False
guilty       layer_8        -0.026      1.9

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found
